# NB06. Interventional Fidelity (C5)

**목적**: 스코어카드의 빈 구조가 제공하는 개선 경로가 실제로 작동하는지 검증한다. "이 변수를 개선하면 점수가 올라간다"는 스코어카드의 처방이 BB 모형에서도 유효한가?

**Dependencies**: NB01 (datasets), NB02 (bb model, bb_shap, train_val_idx)

**Outputs** (`outputs/NB06/`):
- `interventional_results.json`: DA, Spearman, IR, ENI per method

**지표**:
- DA (Directional Accuracy): 스코어카드 예측 방향이 맞는 비율
- Spearman rho: 예측-실제 개선량 순위 상관
- IR (Improvement Ratio): actual/predicted 비율
- ENI (Effort-Normalized Improvement): 표준화 노력 대비 실제 개선

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, json, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from scipy.stats import spearmanr

from decentra._utils import transform_logit_to_score, logit
from decentra.surrogate import TreeSurrogate, EBMSurrogate, LinearSurrogate, BinningSurrogate

NB01_DIR = Path('../outputs/NB01')
NB02_DIR = Path('../outputs/NB02')
OUT_DIR = Path('../outputs/NB06')
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB01_DIR / 'datasets.pkl', 'rb') as f:
    datasets = pickle.load(f)

# Load BB models (need predict_proba for interventional check)
bb_models = {}
for name in datasets:
    bb_models[name] = lgb.Booster(model_file=str(NB02_DIR / f'bb_model_{name}.txt'))

bb = {}
for name in datasets:
    bb[name] = {
        'score_train': np.load(NB02_DIR / f'bb_score_train_{name}.npy'),
        'score_test': np.load(NB02_DIR / f'bb_score_test_{name}.npy'),
        'prob': np.load(NB02_DIR / f'bb_prob_{name}.npy'),
    }
    with open(NB02_DIR / f'train_val_idx_{name}.pkl', 'rb') as f:
        bb[name]['tv_idx'] = pickle.load(f)

N_SIC = 500  # subsample for speed
N_JOBS_EBM = 4
print('Ready')

Ready


## 1. Interventional Fidelity Computation

ScorecardModel의 빈 구조에서 adverse feature의 인접 개선 빈으로 이동 시, BB의 score가 실제로 개선되는지 측정한다. 기존 archive NB09의 `interventional_fidelityorecard` 로직을 재사용하되, DA(not DC) 용어를 사용한다.

In [2]:
def compute_interventional(sm, bb_model, X_eval, feature_stds):
    """Interventional fidelity: scorecard bin movement vs BB actual change.
    
    For each (sample, adverse feature), find improving bin, move x there,
    measure BB score change.
    
    Returns dict with DA, Spearman, IR, ENI, n_pairs.
    """
    X_arr = np.asarray(X_eval)
    
    # BB scores (higher = better, using transform_logit_to_score)
    bb_prob = bb_model.predict(X_arr)
    bb_score = transform_logit_to_score(bb_prob)
    
    # Scorecard contributions (score space: < 0 = adverse)
    contribs = sm.contributions(X_eval)
    
    delta_s_list, delta_b_list, effort_list = [], [], []
    
    for feat in sm.features:
        j = feat.index
        bins = feat.bins
        if len(bins) < 2:
            continue
        
        std_x = feature_stds.get(feat.name, 1.0)
        if std_x < 1e-10:
            continue
        
        for i in range(len(X_arr)):
            if contribs[i, j] >= 0:  # not adverse
                continue
            
            x_val = X_arr[i, j]
            
            # Find current bin
            current_idx = None
            for b_idx, br in enumerate(bins):
                if br.contains(np.array([x_val]))[0]:
                    current_idx = b_idx
                    break
            if current_idx is None:
                continue
            
            current_score = bins[current_idx].score
            
            # Find nearest improving bin (adjacent first)
            best_target = None
            for dist in range(1, len(bins)):
                for direction in [-1, 1]:
                    t_idx = current_idx + direction * dist
                    if t_idx < 0 or t_idx >= len(bins):
                        continue
                    if bins[t_idx].score <= current_score:
                        continue
                    
                    if direction == 1:
                        x_new = bins[t_idx].lower
                    else:
                        x_new = np.nextafter(bins[t_idx].upper, -np.inf)
                    
                    if np.isinf(x_new):
                        continue
                    
                    if best_target is None or abs(x_new - x_val) < abs(best_target[2] - x_val):
                        best_target = (t_idx, bins[t_idx], x_new)
                
                if best_target is not None:
                    break
            
            if best_target is None:
                continue
            
            delta_s = best_target[1].score - current_score
            if delta_s <= 1e-8:
                continue
            
            # BB actual change
            x_mod = X_arr[i].copy()
            x_mod[j] = best_target[2]
            new_prob = bb_model.predict(x_mod.reshape(1, -1))[0]
            new_score = transform_logit_to_score(np.array([new_prob]))[0]
            delta_b = float(new_score - bb_score[i])
            
            delta_s_list.append(delta_s)
            delta_b_list.append(delta_b)
            effort_list.append(abs(best_target[2] - x_val) / std_x)
    
    if len(delta_s_list) < 3:
        return {'DA': np.nan, 'Spearman': np.nan, 'IR': np.nan,
                'ENI': np.nan, 'n_pairs': len(delta_s_list)}
    
    ds = np.array(delta_s_list)
    db = np.array(delta_b_list)
    eff = np.array(effort_list)
    
    da = float(np.mean(db > 0))
    sp = float(spearmanr(ds, db)[0])
    ir = float(np.mean(db / ds))
    
    valid_eni = eff > 0.01
    eni = float(np.mean(db[valid_eni] / eff[valid_eni])) if valid_eni.sum() > 0 else np.nan
    
    return {'DA': round(da, 4), 'Spearman': round(sp, 4), 'IR': round(ir, 4),
            'ENI': round(eni, 4), 'n_pairs': len(ds)}


# Methods to evaluate
SIC_METHODS = {
    'Tree-d1':      lambda: TreeSurrogate(max_depth=1, verbose=0),
    'Tree-d1+mono': lambda: TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0),
    'EBM':          lambda: EBMSurrogate(interactions=0, n_jobs=N_JOBS_EBM),
    'Ridge':        lambda: LinearSurrogate(method='ridgecv'),
    'OptBin+Ridge': lambda: BinningSurrogate(max_n_bins=10),
}

all_results = {}

for data_name in datasets:
    d = datasets[data_name]
    X_train, X_test = d['X_train'], d['X_test']
    feature_names = d['feature_names']
    y_score_train = bb[data_name]['score_train']
    
    tv = bb[data_name]['tv_idx']
    X_tr = X_train.loc[tv['train_idx']]
    X_val = X_train.loc[tv['val_idx']]
    y_tr = y_score_train[X_train.index.get_indexer(tv['train_idx'])]
    y_val = y_score_train[X_train.index.get_indexer(tv['val_idx'])]
    
    # Feature stds from training data
    feature_stds = {col: float(X_train[col].std()) for col in X_train.columns}
    
    # Subsample test for speed
    np.random.seed(317)
    sic_idx = np.random.choice(len(X_test), min(N_SIC, len(X_test)), replace=False)
    X_sic = X_test.iloc[sic_idx]
    
    print(f'\n{"="*70}')
    print(f'  {data_name}: {len(X_sic)} samples for interventional fidelity')
    print(f'{"="*70}')
    
    all_results[data_name] = {}
    
    for method_name, make_surr in SIC_METHODS.items():
        t0 = time.time()
        surr = make_surr()
        
        if isinstance(surr, TreeSurrogate):
            surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        elif isinstance(surr, BinningSurrogate):
            surr.fit(X=X_train, y_logit=y_score_train, feature_names=feature_names)
        else:
            surr.fit(X=X_train, y_logit=y_score_train)
        
        # Convert to ScorecardModel
        sm = surr.to_scorecard_model(
            X_train, y_binary=np.array(d['y_train']),
            feature_names=feature_names,
            max_bins_per_feature=10, min_bin_ratio=0.01
        )
        
        result = compute_interventional(sm, bb_models[data_name], X_sic, feature_stds)
        elapsed = time.time() - t0
        result['time_s'] = round(elapsed, 1)
        
        all_results[data_name][method_name] = result
        print(f'  {method_name:16s}  DA={result["DA"]:.3f}  Spear={result["Spearman"]:.3f}  '
              f'IR={result["IR"]:.2f}  ENI={result["ENI"]:.1f}  '
              f'pairs={result["n_pairs"]}  ({elapsed:.0f}s)')


  GMSC: 500 samples for interventional fidelity


  Tree-d1           DA=0.678  Spear=0.446  IR=1.17  ENI=50.8  pairs=1474  (3s)


  Tree-d1+mono      DA=0.620  Spear=0.449  IR=0.53  ENI=37.4  pairs=1630  (4s)


  EBM               DA=0.460  Spear=0.138  IR=0.39  ENI=13.0  pairs=1706  (39s)


  Ridge             DA=0.193  Spear=-0.411  IR=-2699.30  ENI=-151.3  pairs=2168  (1s)


...

  Tree-d1           DA=0.661  Spear=0.481  IR=1.84  ENI=8.4  pairs=6926  (10s)


  Tree-d1+mono      DA=0.640  Spear=0.438  IR=0.14  ENI=7.8  pairs=7426  (11s)


  EBM               DA=0.323  Spear=0.215  IR=0.58  ENI=4.1  pairs=10146  (421s)


  Ridge             DA=0.308  Spear=0.202  IR=-0.22  ENI=2.5  pairs=10063  (5s)


  OptBin+Ridge      DA=0.352  Spear=0.293  IR=7.50  ENI=19.8  pairs=9221  (9s)


In [3]:
with open(OUT_DIR / 'interventional_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved to {OUT_DIR}/')

Saved to ..\outputs\NB06/
